# Snow Depth Estimation using XGBoost

This notebook estimates snow depth using XGBoost regression trained on geospatial, terrain, environmental, and temporal features.

**Dataset:** ~224,000 samples  
**Features after engineering:** 40  
**Target variable:** SnowDepth

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import shap

import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset

In [ ]:
DATA_PATH = "../data/final_dataset_without_modis_with_features.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

## 3. Dataset Inspection

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 4. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing)

## 5. Date Feature Extraction

In [ ]:
df['date'] = pd.to_datetime(df['date'])

df['year']      = df['date'].dt.year
df['month']     = df['date'].dt.month
df['day']       = df['date'].dt.day
df['dayofyear'] = df['date'].dt.dayofyear

print("Date features extracted: year, month, day, dayofyear")

## 6. Data Preprocessing

In [ ]:
# Drop Station_ID and date after extracting date features
df = df.drop(columns=['Station_ID', 'date'])

# Median imputation for numeric missing values
numeric_cols = df.select_dtypes(include='number').columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Missing values after imputation:", df.isnull().sum().sum())

## 7. Feature Engineering

In [ ]:
df['Elevation_Slope']    = df['elevation'] * df['slope']
df['Tree_Elevation']     = df['Percent_Tree_Cover'] * df['elevation']
df['Terrain_Ruggedness'] = df['tri'] * df['slope']

print("Engineered features added: Elevation_Slope, Tree_Elevation, Terrain_Ruggedness")

## 8. Feature and Target Definition

In [ ]:
TARGET = 'SnowDepth'

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"Features: {X.shape[1]}")
print(f"Samples:  {X.shape[0]}")

## 9. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")

## 10. XGBoost Model Training

In [ ]:
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

model.fit(X_train, y_train)
print("Model training complete.")

## 11. Predictions

In [ ]:
y_pred = model.predict(X_test)

print("Sample Predictions (first 5):")
for actual, predicted in zip(y_test.values[:5], y_pred[:5]):
    print(f"  Actual: {actual:.2f}  |  Predicted: {predicted:.2f}")

## 12. Evaluation

In [ ]:
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

## 13. Feature Importance

In [ ]:
importances = model.feature_importances_
indices     = importances.argsort()[::-1][:15]
top_features    = [X.columns[i] for i in indices]
top_importances = importances[indices]

plt.figure(figsize=(10, 6))
sns.barplot(x=top_importances, y=top_features, palette='viridis')
plt.xlabel('Feature Importance Score')
plt.ylabel('Feature')
plt.title('Top 15 Feature Importances (XGBoost)')
plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=150)
plt.show()

## 14. Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect Fit')
plt.xlabel('Actual Snow Depth')
plt.ylabel('Predicted Snow Depth')
plt.title('Actual vs Predicted Snow Depth')
plt.legend()
plt.tight_layout()
plt.savefig('../results/actual_vs_predicted.png', dpi=150)
plt.show()

## 15. SHAP Explainability

In [ ]:
# Use a sample of 100 test observations for SHAP analysis
X_shap = X_test.iloc[:100]

explainer   = shap.Explainer(model)
shap_values = explainer(X_shap)

shap.summary_plot(shap_values, X_shap, plot_type='bar', show=True)

In [ ]:
shap.summary_plot(shap_values, X_shap, show=True)

## 16. Conclusion

This project trained an XGBoost regression model to estimate snow depth using terrain, vegetation, and temporal features.

**Results (80/20 random train-test split):**

| Metric | Score |
|---|---|
| MAE | 60.02 |
| RMSE | 83.87 |
| R² | 0.9711 |

The model explains approximately 97.11% of the variance in the test set under the random split methodology. Feature importance and SHAP analysis were used to understand which variables most influence snow depth predictions.